# Merge Soil and Meteorological Data
## Overview
This notebook merges cleaned soil and meteorological (MET) data by **aligning timestamps**.  
The merged dataset will be used for further analysis and modeling.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
def load_data(station_id=1):
    file_path_soil = Path(f"../EDA_2025/cleaned_soil_data/SM_{station_id}_cleaned.csv")
    file_path_met = Path(f"../EDA_2025/cleaned_met_data/MET_{station_id}_cleaned.csv")
    df_soil = pd.read_csv(file_path_soil, parse_dates=["Date"], index_col="Date")
    df_met = pd.read_csv(file_path_met, parse_dates=["Date"], index_col="Date")
    return df_soil, df_met

In [3]:
df_soil, df_met = load_data(station_id=5)

# Preview loaded data
print("Soil Data:")
print(df_soil.head())
print("\nMET Data:")
print(df_met.head())

Soil Data:
                     Ppt  SWC_5  SWC_10  SWC_20  SWC_50   T_5  T_10  T_20  \
Date                                                                        
2015-01-01 00:00:00  0.0  0.232   0.266     NaN   0.266  4.35  5.24   0.0   
2015-01-01 01:00:00  0.0  0.232   0.266     NaN   0.266  4.25  5.16   0.0   
2015-01-01 02:00:00  0.0  0.232   0.266     NaN   0.266  4.21  5.09   0.0   
2015-01-01 03:00:00  0.0  0.232   0.265     NaN   0.266  4.19  5.06   0.0   
2015-01-01 04:00:00  0.0  0.232   0.265     NaN   0.266  4.18  5.00   0.0   

                      T_50  Flag  
Date                              
2015-01-01 00:00:00  12.40   176  
2015-01-01 01:00:00  12.37   176  
2015-01-01 02:00:00  12.35   176  
2015-01-01 03:00:00  12.31   176  
2015-01-01 04:00:00  12.31   176  

MET Data:
                     Ppt   Tair    RH  Wind speed  Wind direction  Srad
Date                                                                   
2015-01-01 00:00:00  0.0 -0.632  84.5       1.102

In [4]:
print(df_soil.shape)
print(df_met.shape)

(59161, 10)
(58441, 6)


In [5]:
def merge_datasets(df_soil, df_met):
    """
    Merge soil and meteorological data
    
    Args:
        df_soil: Soil dataset with DateTime index
        df_met: Meteorological dataset with DateTime index
    
    Returns:
        Merged DataFrame with aligned timestamps
    """
    
    # Inner join to keep only matching timestamps
    merged_df = pd.merge(
        df_soil,
        df_met,
        how='inner',  
        left_index=True,
        right_index=True,
        suffixes=('_soil', '_met')
    )
    
    # Validate merge quality
    print(f"Merged dataset contains {len(merged_df)} hourly records")
    print(f"Time range: {merged_df.index.min()} to {merged_df.index.max()}")
    
    return merged_df.sort_index()

In [6]:
merged_df = merge_datasets(df_soil, df_met)

Merged dataset contains 58441 hourly records
Time range: 2015-01-01 00:00:00 to 2021-09-01 00:00:00


In [7]:
merged_df.head(5)

,Ppt_soil,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag,Ppt_met,Tair,RH,Wind speed,Wind direction,Srad
Date,,,,,,,,,,,,,,,,
2015-01-01 00:00:00,0.0,0.232,0.266,NaN,0.266,4.35,5.24,0.0,12.40,176,0.0,-0.632,84.5,1.102,359.60,0.00
2015-01-01 01:00:00,0.0,0.232,0.266,NaN,0.266,4.25,5.16,0.0,12.37,176,0.0,-0.542,84.6,0.874,14.88,0.15
2015-01-01 02:00:00,0.0,0.232,0.266,NaN,0.266,4.21,5.09,0.0,12.35,176,0.0,-0.439,83.4,1.178,22.20,0.27
2015-01-01 03:00:00,0.0,0.232,0.265,NaN,0.266,4.19,5.06,0.0,12.31,176,0.0,-0.328,83.0,0.785,28.02,0.26
2015-01-01 04:00:00,0.0,0.232,0.265,NaN,0.266,4.18,5.00,0.0,12.31,176,0.0,-0.311,91.0,0.811,5.05,0.04


In [8]:
merged_df.tail(5)

,Ppt_soil,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag,Ppt_met,Tair,RH,Wind speed,Wind direction,Srad
Date,,,,,,,,,,,,,,,,
2021-08-31 20:00:00,0.0,0.175,0.171,0.192,0.185,33.26,32.34,31.01,28.62,0,0.0,29.81,59.85,1.728,152.7,0.01
2021-08-31 21:00:00,0.0,0.174,0.171,0.192,0.185,32.62,32.13,31.14,28.59,0,0.0,28.51,64.15,2.122,147.9,0.00
2021-08-31 22:00:00,0.0,0.174,0.170,0.192,0.185,32.03,31.85,31.18,28.59,0,0.0,27.77,68.13,2.449,154.5,0.00
2021-08-31 23:00:00,0.0,0.173,0.170,0.192,0.185,31.50,31.54,31.16,28.59,0,0.0,27.35,70.85,2.817,163.1,0.00
2021-09-01 00:00:00,0.0,0.172,0.169,0.192,0.185,31.02,31.23,31.07,28.60,0,0.0,26.74,76.16,2.837,170.5,0.00


In [9]:
print("\nColumn Summary:")
merged_df.dtypes


Column Summary:


Ppt_soil          float64
SWC_5             float64
SWC_10            float64
SWC_20            float64
SWC_50            float64
T_5               float64
T_10              float64
T_20              float64
T_50              float64
Flag                int64
Ppt_met           float64
Tair              float64
RH                float64
Wind speed        float64
Wind direction    float64
Srad              float64
dtype: object

In [10]:
merged_df.describe()

,Ppt_soil,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag,Ppt_met,Tair,RH,Wind speed,Wind direction,Srad
count,57968.000000,57844.000000,57859.000000,47887.000000,57861.000000,57968.000000,57968.000000,57968.000000,57968.000000,58441.000000,57865.000000,57864.000000,57865.000000,57865.000000,57865.000000,57817.000000
mean,0.077529,0.249706,0.262953,0.254246,0.244896,21.601010,21.614229,18.196719,21.453011,172.994764,0.077664,19.556927,58.237905,2.483684,165.160624,206.356856
std,0.809306,0.063492,0.063561,0.054316,0.063624,8.570405,8.096240,10.698670,6.049467,1296.826496,0.809959,8.962607,31.322665,1.403903,87.833638,295.827974
min,0.000000,0.135000,0.147000,0.169000,0.167000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-40.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.189000,0.201000,0.201000,0.187000,14.850000,15.090000,11.710000,15.920000,0.000000,0.000000,13.910000,36.840000,1.483000,131.800000,0.000000
50%,0.000000,0.252000,0.269000,0.256000,0.235000,21.870000,21.900000,19.570000,21.260000,0.000000,0.000000,20.830000,62.610000,2.427000,166.700000,10.160000
75%,0.000000,0.302000,0.317000,0.296000,0.291000,28.262500,28.110000,26.830000,26.890000,0.000000,0.000000,25.540000,86.000000,3.431000,189.900000,369.900000
max,40.640000,0.444000,0.457000,0.465000,0.488000,44.100000,40.980000,38.320000,32.600000,13232.000000,40.640000,42.500000,100.000000,8.260000,360.000000,1103.000000


In [11]:
def save_merged_data(df, station_id, output_dir="merged_data"):
    """
    Save merged data to CSV.
    
    Args:
        df (pd.DataFrame): Merged soil and met data.
        station_id (int): Station ID (1-6).
        output_dir (str): Output directory path.
    """
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/merged_station_{station_id}.csv"
    df.to_csv(output_path)
    print(f"Saved merged data to: {output_path}")

    return df

In [12]:
#save_merged_data(merged_df, 5)